# ❤️ التنبؤ بمخاطر أمراض القلب (Heart Disease Risk Classifier)
### مشروع B5 — مسار تعلم الآلة (Machine Learning Track)

---
## 🎯 المشكلة التجارية (Business Problem)
عيادة عايزة **أداة فرز (Screening)** تتنبأ باحتمال إصابة المريض بمرض قلب من فحوصاته الأساسية،
عشان توجّه الأطباء للحالات الخطيرة بدري. **قرار طبي حسّاس → التفسير (Interpretability) مهم جداً.**

**نوع المشكلة:** تصنيف ثنائي (مريض / سليم).

## 📦 ما الذي يثبته المشروع
كشف **تسريب البيانات (Data Leakage)** من ميزة مشتقة · Pipeline · مقارنة موديلات ·
تقييم (ROC-AUC) · **تفسير العوامل الطبية (Feature Importance)**.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "ml/b5_heart_disease"          # مسار المشروع داخل portfolio/
PKGS    = ["xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| التصنيف (LogReg, RF) | ISLR (ch.4) / Géron | أساس التنبؤ الثنائي |
| **Data Leakage من ميزة مشتقة** | Huyen / Géron | ميزة "مثالية" غالباً غش — لازم تكتشفها |
| ROC-AUC + Confusion Matrix | ISLR (ch.4) | تقييم مع أهمية الحالات الإيجابية |
| Feature Importance | Géron / Molnar | في الطب لازم تعرف "ليه" |

> 🎯 **بيُستخدم في الواقع:** أنظمة الدعم الطبي، فرز المرضى، التأمين الصحي، الطب الوقائي.
> ⚠️ **تنبيه أخلاقي:** أداة فرز مساعِدة فقط — مش بديل عن تشخيص الطبيب.


## 0️⃣ المكتبات

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); np.random.seed(42)
print('ready ✓')

## 1️⃣ تحميل واستكشاف البيانات (EDA)

In [ ]:
df = pd.read_csv('data/health_risk_data.csv')
# TODO: shape + معدل المرض + الارتباط مع الهدف
print(df['heart_disease'].mean())

## 2️⃣ ⚠️ كشف تسريب البيانات (Data Leakage)
لاحظ إن `risk_score` ارتباطه بالهدف **عالي جداً (~0.80)**. ده مش ميزة حقيقية — ده **درجة خطورة محسوبة
من النتيجة نفسها**. لو سِبناها، الموديل هيغش وهيفشل في الواقع. **نشيلها.**

In [ ]:
# TODO: شيل العمود المسرّب risk_score + المعرّفات، وحدّد X و y
LEAKY = ['risk_score']
X = ...
y = ...

## 3️⃣ المعالجة والتقسيم (Pipeline)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# TODO: حدّد الأعمدة، split stratified، ColumnTransformer
...

## 4️⃣ مقارنة الموديلات (Cross-Validation, ROC-AUC)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
# TODO: قارن 3 موديلات بـ 5-fold CV على ROC-AUC
...

## 5️⃣ التقييم النهائي

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, RocCurveDisplay
# TODO: درّب أفضل موديل، قيّم، وارسم confusion matrix + ROC
best = ...

## 6️⃣ تفسير العوامل الطبية (Feature Importance)

In [ ]:
rf = Pipeline([('p',pre),('m',models['RandomForest'])]).fit(X_tr, y_tr)
names = rf.named_steps['p'].get_feature_names_out()
imp = pd.Series(rf.named_steps['m'].feature_importances_, index=names).sort_values().tail(10)
imp.plot(kind='barh', figsize=(7,4), title='Top risk factors'); plt.tight_layout(); plt.show()

## 7️⃣ الخلاصة والتوصيات (Conclusion)
- **درس التسريب:** شيلنا `risk_score` لأنها مشتقة من النتيجة — لو سِبناها كانت الدقة "وهمية".
- **النتيجة:** بعد إزالة التسريب، الموديل حقّق ROC-AUC واقعي وكويس بالعوامل الطبية الحقيقية.
- **أهم العوامل:** السن، الكوليسترول، ضغط الدم الانقباضي — متوافقة مع الطب المعروف ✓ (ثقة في الموديل).
- **التوصية:** استخدامه كأداة فرز مساعِدة توجّه الانتباه للحالات عالية الخطورة، مع تأكيد الطبيب دائماً.
- **الخطوة القادمة:** معايرة الاحتمالات + تحليل بـ SHAP لكل مريض على حدة.

> ✅ **اللي اتعلمته:** كشف الـ leakage من ميزة مشتقة، Pipeline، مقارنة موديلات، وتفسير طبي.
